# Stage 1 (Part 1) – Data Preparation

Cleans the raw IBM Telco file (7,043 customers, 21 columns) and creates the train/test sets used by all later notebooks. Everything that learns from data is fitted on the training set only. No resampling.

**Outputs:** `data/processed/train.csv`, `test.csv` · Tables 00–03 in `outputs/tables/` · `src/features.py`

## A. Setup
Loads the project settings from `config.py` (paths, random seed, test size) so every notebook and script uses the same values.

In [1]:
# Cell 1 - Imports and project paths
import sys
from pathlib import Path

import pandas as pd

# Find the project root, whether this notebook sits in src/ or notebooks/
ROOT = Path.cwd()
if ROOT.name in ("src", "notebooks"):
    ROOT = ROOT.parent
sys.path.append(str(ROOT))

import config as C

print("Project root:      ", ROOT)
print("Raw file exists:   ", C.RAW_CSV.exists())
print("Random seed:       ", C.RANDOM_STATE)
print("Test size:         ", C.TEST_SIZE)

Project root:       /Users/itsupport/MScThesis/churn-thesis
Raw file exists:    True
Random seed:        42
Test size:          0.2


## B. Load and profile the raw data
The raw CSV is never edited. Every column is profiled for type, number of unique values, missing values and blank text.

In [2]:
# Cell 2 - Load the raw data and profile every column
df_raw = pd.read_csv(C.RAW_CSV)          # the original file is never edited
flow = [("Raw IBM Telco file", len(df_raw))]   # data-flow log, saved as a table later

print("Rows, columns:", df_raw.shape)

overview = pd.DataFrame({
    "dtype": df_raw.dtypes.astype(str),
    "unique_values": df_raw.nunique(),
    "missing (NaN)": df_raw.isna().sum(),
    "blank text": df_raw.apply(lambda s: (s.astype(str).str.strip() == "").sum()),
    "examples": df_raw.apply(lambda s: ", ".join(map(str, s.unique()[:3]))),
})
overview

Rows, columns: (7043, 21)


,dtype,unique_values,missing (NaN),blank text,examples
customerID,str,7043,0,0,"7590-VHVEG, 5575-GNVDE, 3668-QPYBK"
gender,str,2,0,0,"Female, Male"
SeniorCitizen,int64,2,0,0,"0, 1"
Partner,str,2,0,0,"Yes, No"
Dependents,str,2,0,0,"No, Yes"
tenure,int64,73,0,0,"1, 34, 2"
PhoneService,str,2,0,0,"No, Yes"
MultipleLines,str,3,0,0,"No phone service, No, Yes"
InternetService,str,3,0,0,"DSL, Fiber optic, No"
OnlineSecurity,str,3,0,0,"No, Yes, No internet service"


**Findings from profiling**
1. **TotalCharges** is stored as text because 11 rows contain a blank space, so they are not detected as missing (NaN).
2. Six internet add-on columns contain **"No internet service"**, and MultipleLines contains **"No phone service"**. These repeat information already in InternetService and PhoneService.
3. **SeniorCitizen** is coded 0/1, while every other yes/no column uses text.
4. **tenure** has 73 unique values (0–72 months), so customers with tenure = 0 exist.
5. **customerID** is unique (7,043 values): no duplicate customers.
6. **Churn** is stored as Yes/No text.

## C. Cleaning
### Investigation: the 11 customers with blank TotalCharges

In [3]:
# Cell 3 - Investigate the 11 customers with blank TotalCharges
blank = df_raw["TotalCharges"].astype(str).str.strip() == ""

print("Blank TotalCharges:", blank.sum())
print("Customers with tenure = 0:", (df_raw["tenure"] == 0).sum())
print("Are they the same customers?", (blank == (df_raw["tenure"] == 0)).all())
print("Churners among them:", (df_raw.loc[blank, "Churn"] == "Yes").sum())

df_raw.loc[blank, ["customerID", "tenure", "MonthlyCharges", "TotalCharges",
                   "Contract", "Churn"]]

Blank TotalCharges: 11
Customers with tenure = 0: 11
Are they the same customers? True
Churners among them: 0


,customerID,tenure,MonthlyCharges,TotalCharges,Contract,Churn
488,4472-LVYGI,0,52.55,,Two year,No
753,3115-CZMZD,0,20.25,,Two year,No
936,5709-LVOEQ,0,80.85,,Two year,No
1082,4367-NUYAO,0,25.75,,Two year,No
1340,1371-DWPAZ,0,56.05,,Two year,No
3331,7644-OMVMY,0,19.85,,Two year,No
3826,3213-VVOLG,0,25.35,,Two year,No
4380,2520-SGTTA,0,20.00,,Two year,No
5218,2923-ARZLG,0,19.70,,One year,No
6670,4075-WKNIU,0,73.35,,Two year,No


**Findings:** All 11 blank TotalCharges belong to the 11 customers with **tenure = 0** (10 on two-year and 1 on one-year contracts), and **none of them churned**. They joined in the current month and have not been billed yet. The values are **structurally missing**, not random gaps or noise.

### Decision 1: Remove the 11 customers with blank TotalCharges
7,043 → 7,032. They have tenure 0 and were never billed, so they are not yet at risk of churning; median imputation would invent charges they never paid. Keeping them would change the churn rate only from 26.58% to 26.54%.

In [4]:
# Cell 4 - Convert TotalCharges to a number and drop the 11 tenure-0 customers
df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")   # blanks become NaN

# Robustness check: churn rate if we had kept them (with TotalCharges = 0)
churn_if_kept = (df["Churn"] == "Yes").mean()

df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)
flow.append(("Removed: blank TotalCharges (tenure = 0)", len(df)))

print("Rows left:              ", len(df))
print("Missing values now:     ", df.isna().sum().sum())
print(f"Churn if kept (TC = 0): {churn_if_kept:.2%}")
print(f"Churn after dropping:   {(df['Churn'] == 'Yes').mean():.2%}")

Rows left:               7032
Missing values now:      0
Churn if kept (TC = 0): 26.54%
Churn after dropping:   26.58%


### Decision 2: Check whether any other rows should be removed
Each row is tested for duplicates, impossible values, billing inconsistencies, contradictions between related columns, and extreme values (IQR rule).

In [5]:
# Cell 5 - Are there any other rows that should be removed?
addons = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
          "TechSupport", "StreamingTV", "StreamingMovies"]
no_internet = df["InternetService"] == "No"
no_phone = df["PhoneService"] == "No"
ratio = df["TotalCharges"] / (df["tenure"] * df["MonthlyCharges"])

def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

checks = pd.DataFrame([
    ("Duplicate customerID",                       df[C.ID_COL].duplicated().sum()),
    ("Identical rows apart from customerID",       df.drop(columns=C.ID_COL).duplicated().sum()),
    ("tenure <= 0 or > 72",                        ((df["tenure"] <= 0) | (df["tenure"] > 72)).sum()),
    ("MonthlyCharges <= 0",                        (df["MonthlyCharges"] <= 0).sum()),
    ("TotalCharges < MonthlyCharges",              (df["TotalCharges"] < df["MonthlyCharges"]).sum()),
    ("Billing ratio outside 0.5-1.5",              ((ratio < 0.5) | (ratio > 1.5)).sum()),
    ("No internet, but an add-on is Yes/No",       (no_internet & (df[addons] != "No internet service").any(axis=1)).sum()),
    ("Internet, but add-on says 'No internet'",    (~no_internet & (df[addons] == "No internet service").any(axis=1)).sum()),
    ("PhoneService and MultipleLines disagree",    (no_phone != (df["MultipleLines"] == "No phone service")).sum()),
    ("IQR outliers: tenure",                       iqr_outliers(df["tenure"])),
    ("IQR outliers: MonthlyCharges",               iqr_outliers(df["MonthlyCharges"])),
    ("IQR outliers: TotalCharges",                 iqr_outliers(df["TotalCharges"])),
], columns=["Check", "Rows flagged"])

print(f"Billing ratio TotalCharges / (tenure x MonthlyCharges): "
      f"median {ratio.median():.2f}, min {ratio.min():.2f}, max {ratio.max():.2f}")
checks

Billing ratio TotalCharges / (tenure x MonthlyCharges): median 1.00, min 0.69, max 1.57


,Check,Rows flagged
0,Duplicate customerID,0
1,Identical rows apart from customerID,22
2,tenure <= 0 or > 72,0
3,MonthlyCharges <= 0,0
4,TotalCharges < MonthlyCharges,0
5,Billing ratio outside 0.5-1.5,2
6,"No internet, but an add-on is Yes/No",0
7,"Internet, but add-on says 'No internet'",0
8,PhoneService and MultipleLines disagree,0
9,IQR outliers: tenure,0


**Findings**
- No duplicate IDs, impossible values, contradictions between related columns or IQR outliers.
- **22 identical profiles: kept.** They are all tenure-1 customers on basic plans with different customer IDs; with 19 mostly yes/no columns, real customers can share a profile.
- **2 billing ratios outside 0.5–1.5: kept.** Both have 2–3 months tenure and low charges, consistent with a move to a cheaper plan.
- Billing ratio median 1.00 (min 0.69, max 1.57): TotalCharges agrees with tenure × MonthlyCharges.

**Decision:** no further rows removed. Final dataset: **7,032 customers**. Saved as Table 01.

### Decision 3: Encode the target and SeniorCitizen
Churn → 0/1 (1 = churned); SeniorCitizen 0/1 → No/Yes. The class imbalance is recorded, not corrected (no resampling).

In [6]:
# Cell 6 - Target to 0/1, SeniorCitizen to No/Yes, class balance
df[C.TARGET] = (df[C.TARGET] == "Yes").astype(int)
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

n_churn = df[C.TARGET].sum()
churn_rate = df[C.TARGET].mean()

print("Churn values:        ", sorted(df[C.TARGET].unique().tolist()))
print("SeniorCitizen values:", sorted(df["SeniorCitizen"].unique()))
print(f"Churned: {n_churn} of {len(df)}  ({churn_rate:.2%})")
print(f"Class ratio (stayed : churned) = {(1 - churn_rate) / churn_rate:.1f} : 1")

Churn values:         [0, 1]
SeniorCitizen values: ['No', 'Yes']
Churned: 1869 of 7032  (26.58%)
Class ratio (stayed : churned) = 2.8 : 1


**Findings:** 1,869 of 7,032 customers churned (**26.58%**), a **2.8 : 1** class ratio. This is a mild imbalance, so the split is stratified. Accuracy alone would be misleading: predicting "no churn" for everyone already scores 73%.

## D. Feature preparation
### Step 4: Feature types
Classifies the 19 input features as numeric, binary or multi-class.

In [7]:
# Cell 7 - Feature types and the data-quality table
features = df.drop(columns=[C.ID_COL, C.TARGET])
numeric = features.select_dtypes("number").columns.tolist()
categorical = features.select_dtypes(exclude="number").columns.tolist()
binary = [c for c in categorical if features[c].nunique() == 2]
multi = [c for c in categorical if features[c].nunique() > 2]

def describe(col):
    s = features[col]
    if col in numeric:
        return f"{s.min():g} to {s.max():g}"
    return ", ".join(sorted(s.unique()))

feature_types = pd.DataFrame({
    "type": ["numeric"] * len(numeric) + ["binary"] * len(binary) + ["multi-class"] * len(multi),
    "levels / range": [describe(c) for c in numeric + binary + multi],
}, index=numeric + binary + multi)
feature_types.index.name = "feature"

# Save the quality checks from Cell 5
C.TABLES_DIR.mkdir(parents=True, exist_ok=True)
checks.to_csv(C.TABLES_DIR / "01_data_quality.csv", index=False)

print(f"Numeric: {len(numeric)}   Binary: {len(binary)}   "
      f"Multi-class: {len(multi)}   Total: {features.shape[1]}")
feature_types

Numeric: 3   Binary: 6   Multi-class: 10   Total: 19


,type,levels / range
feature,,
tenure,numeric,1 to 72
MonthlyCharges,numeric,18.25 to 118.75
TotalCharges,numeric,18.8 to 8684.8
gender,binary,"Female, Male"
SeniorCitizen,binary,"No, Yes"
Partner,binary,"No, Yes"
Dependents,binary,"No, Yes"
PhoneService,binary,"No, Yes"
PaperlessBilling,binary,"No, Yes"


**Findings:** 19 features: 3 numeric, 6 binary and 10 multi-class. tenure now runs from 1 to 72. **7 of the 10 multi-class columns are multi-class only because of the redundant third level**; only InternetService, Contract and PaymentMethod are truly multi-class.

### Decision 4: Merge redundant levels
"No internet service" / "No phone service" → "No" for the model features (the information stays in InternetService and PhoneService). The original levels are kept in `df_original` for the EDA. Cramér's V is shown before and after merging.

In [8]:
# Cell 8 - Keep an original copy for EDA, merge redundant levels for features
import numpy as np
from scipy.stats import chi2_contingency

INTERNET_ADDONS = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                   "TechSupport", "StreamingTV", "StreamingMovies"]

df_original = df.copy()                                    # 3 levels - for EDA
df[INTERNET_ADDONS] = df[INTERNET_ADDONS].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

def cramers_v(x, y):
    """Strength of association between two categorical variables (0 = none, 1 = perfect)."""
    table = pd.crosstab(x, y)
    chi2 = chi2_contingency(table, correction=False)[0]
    return np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))

merge_effect = pd.DataFrame(
    [(col,
      cramers_v(df_original[col], df_original[C.TARGET]),
      cramers_v(df[col], df[C.TARGET]))
     for col in INTERNET_ADDONS + ["MultipleLines"]],
    columns=["feature", "V original (3 levels)", "V merged (2 levels)"],
).round(3)

print("Levels after merging:", {c: sorted(df[c].unique()) for c in INTERNET_ADDONS + ["MultipleLines"]})
print("\nChurn rate by OnlineSecurity (original levels):")
print(df_original.groupby("OnlineSecurity")[C.TARGET].agg(customers="size", churn_rate="mean").round(3))
merge_effect

Levels after merging: {'OnlineSecurity': ['No', 'Yes'], 'OnlineBackup': ['No', 'Yes'], 'DeviceProtection': ['No', 'Yes'], 'TechSupport': ['No', 'Yes'], 'StreamingTV': ['No', 'Yes'], 'StreamingMovies': ['No', 'Yes'], 'MultipleLines': ['No', 'Yes']}

Churn rate by OnlineSecurity (original levels):
                     customers  churn_rate
OnlineSecurity                            
No                        3497       0.418
No internet service       1520       0.074
Yes                       2015       0.146


,feature,V original (3 levels),V merged (2 levels)
0,OnlineSecurity,0.347,0.171
1,OnlineBackup,0.292,0.082
2,DeviceProtection,0.281,0.066
3,TechSupport,0.343,0.165
4,StreamingTV,0.230,0.063
5,StreamingMovies,0.231,0.061
6,MultipleLines,0.040,0.040


**Findings:** customers without internet churn at only **7.4%**, compared with 41.8% (no security) and 14.6% (has security). This group inflates the original Cramér's V:
- OnlineSecurity drops from 0.35 to 0.17 and TechSupport from 0.34 to 0.17. Both keep a moderate association.
- Backup, device protection and the streaming services drop to about 0.06–0.08, a weak association.
- MultipleLines is unchanged (0.04).

Much of the add-on effect reported in Telco churn studies is therefore an internet-service effect.

### Decision 5: Create n_services
Number of services each customer holds (1–9). Saved as an optional feature only, because it correlates 0.85 with MonthlyCharges.

In [9]:
# Cell 9 - n_services: number of services each customer holds
SERVICE_FLAGS = ["PhoneService", "MultipleLines"] + INTERNET_ADDONS     # Yes/No columns

def add_n_services(data):
    data = data.copy()
    data["n_services"] = ((data[SERVICE_FLAGS] == "Yes").sum(axis=1)
                          + (data["InternetService"] != "No").astype(int))
    return data

df = add_n_services(df)
df_original["n_services"] = df["n_services"]          # same counts, for the EDA

print("Range:", df["n_services"].min(), "to", df["n_services"].max())
print("Correlation with MonthlyCharges:", round(df["n_services"].corr(df["MonthlyCharges"]), 3))
print("Correlation with tenure:        ", round(df["n_services"].corr(df["tenure"]), 3))
df.groupby("n_services")[C.TARGET].agg(customers="size", churn_rate="mean").round(3)

Range: 1 to 9
Correlation with MonthlyCharges: 0.851
Correlation with tenure:         0.474


,customers,churn_rate
n_services,,
1,1260,0.110
2,857,0.310
3,846,0.449
4,965,0.365
5,921,0.314
6,906,0.256
7,674,0.226
8,395,0.124
9,208,0.053


**Findings:** churn follows an **inverted U**: 11% with 1 service (mostly phone-only customers), a peak of **45% at 3 services**, then down to **5% with all 9**. Once a customer has internet, each extra service lowers churn ("stickiness"). Business implication: move customers from 2–4 services up to 5 or more. Correlation 0.85 with MonthlyCharges and 0.47 with tenure.

### Decision 6: Feature dictionary
Gives every column one role: input, target, identifier, excluded or optional. TotalCharges is excluded as an input because it ≈ tenure × MonthlyCharges (checked with VIF in Cell 12).

In [10]:
# Cell 10 - Feature dictionary: type, levels and role of every column
ROLES = {
    "customerID":   ("identifier", "Unique ID with no predictive meaning"),
    "Churn":        ("target",     "Outcome to predict (1 = churned)"),
    "TotalCharges": ("excluded",   "≈ tenure × MonthlyCharges (median ratio 1.00); kept for EDA and revenue"),
    "n_services":   ("optional",   "Engineered: count of services; r = 0.85 with MonthlyCharges"),
}

def column_type(s):
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 2:
        return "numeric"
    return "binary" if s.nunique() == 2 else "multi-class"

def column_levels(s):
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 2:
        return f"{s.min():g} to {s.max():g}"
    return ", ".join(map(str, sorted(s.unique())))

feature_dict = pd.DataFrame([{
    "feature":        col,
    "type":           column_type(df[col]),
    "levels / range": column_levels(df[col]),
    "role":           ROLES.get(col, ("input", ""))[0],
    "note":           ROLES.get(col, ("input", ""))[1],
} for col in df.columns if col != C.ID_COL])

feature_dict.to_csv(C.TABLES_DIR / "02_feature_dictionary.csv", index=False)

print(feature_dict["role"].value_counts().to_string())
print("\nInput types:", feature_dict.query("role == 'input'")["type"].value_counts().to_dict())
feature_dict

role
input       18
excluded     1
target       1
optional     1

Input types: {'binary': 13, 'multi-class': 3, 'numeric': 2}


,feature,type,levels / range,role,note
0,gender,binary,"Female, Male",input,
1,SeniorCitizen,binary,"No, Yes",input,
2,Partner,binary,"No, Yes",input,
3,Dependents,binary,"No, Yes",input,
4,tenure,numeric,1 to 72,input,
5,PhoneService,binary,"No, Yes",input,
6,MultipleLines,binary,"No, Yes",input,
7,InternetService,multi-class,"DSL, Fiber optic, No",input,
8,OnlineSecurity,binary,"No, Yes",input,
9,OnlineBackup,binary,"No, Yes",input,


**Findings:** 18 input features (13 binary, 3 multi-class, 2 numeric), plus 1 target, 1 identifier, 1 excluded (TotalCharges) and 1 optional (n_services). Saved as Table 02.

## E. Train/test split and checks
### Decision 7: Stratified 80/20 split (seed 42)
From here on, anything that learns from data is fitted on the training set only. The test set is used once, for the final evaluation.

In [11]:
# Cell 11 - Stratified train/test split, balance check, save
from sklearn.model_selection import train_test_split

train, test = train_test_split(
    df,
    test_size=C.TEST_SIZE,
    stratify=df[C.TARGET],
    random_state=C.RANDOM_STATE,
)
train = train.reset_index(drop=True)
test = test.reset_index(drop=True)

flow += [("Training set (80%)", len(train)), ("Test set (20%)", len(test))]

# Are the two sets alike? (they should be, if the split is fair)
balance = pd.DataFrame({
    "train": [train[C.TARGET].mean(), train["tenure"].mean(), train["MonthlyCharges"].mean(),
              (train["Contract"] == "Month-to-month").mean(), (train["InternetService"] == "Fiber optic").mean()],
    "test":  [test[C.TARGET].mean(), test["tenure"].mean(), test["MonthlyCharges"].mean(),
              (test["Contract"] == "Month-to-month").mean(), (test["InternetService"] == "Fiber optic").mean()],
}, index=["Churn rate", "Mean tenure", "Mean MonthlyCharges", "% month-to-month", "% fibre optic"]).round(3)

# Save
C.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
train.to_csv(C.PROCESSED_DIR / "train.csv", index=False)
test.to_csv(C.PROCESSED_DIR / "test.csv", index=False)
pd.DataFrame(flow, columns=["Step", "Customers"]).to_csv(C.TABLES_DIR / "00_data_flow.csv", index=False)

print(f"Train: {len(train)} rows, {train[C.TARGET].sum()} churners")
print(f"Test:  {len(test)} rows, {test[C.TARGET].sum()} churners")
print("Saved to:", C.PROCESSED_DIR)
print(pd.DataFrame(flow, columns=["Step", "Customers"]).to_string(index=False))
balance

Train: 5625 rows, 1495 churners
Test:  1407 rows, 374 churners
Saved to: /Users/itsupport/MScThesis/churn-thesis/data/processed
                                    Step  Customers
                      Raw IBM Telco file       7043
Removed: blank TotalCharges (tenure = 0)       7032
                      Training set (80%)       5625
                          Test set (20%)       1407


,train,test
Churn rate,0.266,0.266
Mean tenure,32.562,31.860
Mean MonthlyCharges,64.999,63.994
% month-to-month,0.548,0.561
% fibre optic,0.441,0.436


**Findings:** train 5,625 customers (1,495 churners), test 1,407 (374 churners); 26.6% churn in both. Mean tenure, mean MonthlyCharges, % month-to-month and % fibre optic differ by about 1 point or less, so the split is fair. Data flow saved as Table 00; the cleaned (not encoded) train and test files are saved in `data/processed/`.

### Decision 8: Check collinearity with VIF (training set only)
Compares VIF for A: all four numeric columns, B: without TotalCharges, C: the final numeric inputs. VIF > 10 is serious.

In [12]:
# Cell 12 - VIF on the training set: confirm the TotalCharges and n_services decisions
from sklearn.linear_model import LinearRegression

def vif_table(data, cols):
    """VIF for each column = 1 / (1 - R²) of a regression on the other columns."""
    vif = {}
    for col in cols:
        others = [c for c in cols if c != col]
        r2 = LinearRegression().fit(data[others], data[col]).score(data[others], data[col])
        vif[col] = 1 / (1 - r2)
    return pd.Series(vif)

feature_sets = {
    "A: all four":             ["tenure", "MonthlyCharges", "TotalCharges", "n_services"],
    "B: without TotalCharges": ["tenure", "MonthlyCharges", "n_services"],
    "C: final inputs":         ["tenure", "MonthlyCharges"],
}

vif = pd.DataFrame({name: vif_table(train, cols) for name, cols in feature_sets.items()})
vif = vif.loc[["TotalCharges", "tenure", "MonthlyCharges", "n_services"]].round(2)
vif.to_csv(C.TABLES_DIR / "03_vif.csv")
vif

,A: all four,B: without TotalCharges,C: final inputs
TotalCharges,10.18,NaN,NaN
tenure,5.89,1.45,1.07
MonthlyCharges,5.31,4.12,1.07
n_services,5.27,4.97,NaN


**Findings**
- With all four columns, **TotalCharges has VIF 10.18** (serious) and tenure 5.89.
- Removing TotalCharges brings tenure down to 1.45, which confirms TotalCharges mostly duplicates tenure × MonthlyCharges.
- **n_services stays near 5 (4.97)** because of its overlap with MonthlyCharges, which confirms it as optional only.
- The final numeric inputs (tenure, MonthlyCharges) have **VIF 1.07**: no collinearity remains.

Saved as Table 03.

### Decision 9: Encoding, fitted on the training set only
`get_X_y()` keeps the input columns; `make_encoder()` scales numeric columns and one-hot encodes categorical ones. Fitted on train, then applied to test.

In [13]:
# Cell 13 - Model-agnostic encoding: get_X_y() and make_encoder(), fitted on train only
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NOT_INPUTS = feature_dict.loc[feature_dict["role"] != "input", "feature"].tolist() + [C.ID_COL]

def get_X_y(data, include_optional=False):
    """Inputs (X) and target (y). Drops the non-input columns from the feature dictionary."""
    drop = [c for c in NOT_INPUTS if not (include_optional and c == "n_services")]
    return data.drop(columns=drop), data[C.TARGET]

def make_encoder(X, scale_numeric=True, drop="if_binary"):
    """Scale numeric columns and one-hot encode categorical ones. Fit on train only."""
    numeric_cols = X.select_dtypes("number").columns.tolist()
    categorical_cols = X.select_dtypes(exclude="number").columns.tolist()
    return ColumnTransformer([
        ("num", StandardScaler() if scale_numeric else "passthrough", numeric_cols),
        ("cat", OneHotEncoder(drop=drop, handle_unknown="ignore", sparse_output=False), categorical_cols),
    ])

X_train, y_train = get_X_y(train)
X_test, y_test = get_X_y(test)

encoder = make_encoder(X_train)
X_train_enc = encoder.fit_transform(X_train)     # learns from train
X_test_enc = encoder.transform(X_test)           # only applies

print("Removed from inputs:     ", NOT_INPUTS)
print("Inputs before encoding:  ", X_train.shape[1])
print("Columns after encoding:   train", X_train_enc.shape, " test", X_test_enc.shape)
print("Missing values:           train", np.isnan(X_train_enc).sum(), " test", np.isnan(X_test_enc).sum())
print("Scaled tenure, MonthlyCharges - train mean:", X_train_enc[:, :2].mean(axis=0).round(3),
      " test mean:", X_test_enc[:, :2].mean(axis=0).round(3))
print("\nEncoded columns:")
print(list(encoder.get_feature_names_out()))

Removed from inputs:      ['TotalCharges', 'Churn', 'n_services', 'customerID']
Inputs before encoding:   18
Columns after encoding:   train (5625, 25)  test (1407, 25)
Missing values:           train 0  test 0
Scaled tenure, MonthlyCharges - train mean: [-0.  0.]  test mean: [-0.029 -0.033]

Encoded columns:
['num__tenure', 'num__MonthlyCharges', 'cat__gender_Male', 'cat__SeniorCitizen_Yes', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__PhoneService_Yes', 'cat__MultipleLines_Yes', 'cat__InternetService_DSL', 'cat__InternetService_Fiber optic', 'cat__InternetService_No', 'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_Yes', 'cat__DeviceProtection_Yes', 'cat__TechSupport_Yes', 'cat__StreamingTV_Yes', 'cat__StreamingMovies_Yes', 'cat__Contract_Month-to-month', 'cat__Contract_One year', 'cat__Contract_Two year', 'cat__PaperlessBilling_Yes', 'cat__PaymentMethod_Bank transfer (automatic)', 'cat__PaymentMethod_Credit card (automatic)', 'cat__PaymentMethod_Electronic check', 'cat__PaymentMet

**Findings:** 18 inputs encode to **25 columns** (2 scaled numeric, 13 binary, and 10 one-hot levels for InternetService, Contract and PaymentMethod). Train and test have identical columns and no missing values. The scaled train mean is exactly 0 and the test mean about −0.03, which shows the scaler learned from the training set only: **no leakage**.

### Check: `src/features.py` gives the same result as this notebook
Expected: five lines of True.

In [14]:
# Cell 14 - Confirm src/features.py reproduces the notebook exactly
import importlib
from src import features as F
importlib.reload(F)                       # picks up any edits to the file

df_check = F.prepare(df_raw)
X_check, _ = F.get_X_y(train)
enc_check = F.make_encoder(X_check).fit(X_check)

print("Same cleaned data:      ", df_check.equals(df))
print("Same input columns:     ", list(X_check.columns) == list(X_train.columns))
print("Same encoded columns:   ", list(enc_check.get_feature_names_out()) == list(encoder.get_feature_names_out()))
print("Same encoded values:    ", np.allclose(enc_check.transform(X_test), X_test_enc))
print("Same feature dictionary:", F.feature_dictionary(df).equals(feature_dict))

Same cleaned data:       True
Same input columns:      True
Same encoded columns:    True
Same encoded values:     True
Same feature dictionary: True


## Summary of Stage 1 decisions

| # | Decision | Evidence |
|---|---|---|
| 1 | Drop 11 tenure-0 customers (7,043 → 7,032) | Structurally missing TotalCharges; churn 26.58% vs 26.54% if kept |
| 2 | No other rows removed | No duplicates, impossible values, contradictions or IQR outliers |
| 3 | Churn → 0/1; SeniorCitizen → No/Yes; no resampling | Churn 26.58%, ratio 2.8 : 1 |
| 4 | Merge redundant levels for features; original levels for EDA | V of OnlineSecurity 0.35 → 0.17 |
| 5 | n_services created as an optional feature | Inverted-U churn; r = 0.85 with MonthlyCharges |
| 6 | TotalCharges excluded as an input | ≈ tenure × MonthlyCharges (median ratio 1.00) |
| 7 | Stratified 80/20 split, seed 42 | 26.6% churn in both sets |
| 8 | Final numeric inputs: tenure, MonthlyCharges | VIF 10.18 → 1.07 |
| 9 | Model-agnostic encoder, fitted on train only | 25 columns; test mean ≈ −0.03 |

**Outputs:** `data/processed/train.csv` (5,625) and `test.csv` (1,407) · Tables 00 data flow, 01 data quality, 02 feature dictionary, 03 VIF · `src/features.py`

**Next:** exploratory analysis of churn – Q2 Who churns? Q3 When do customers churn? Q4 Is every lost customer equally costly?